In [ ]:
import json
import csv
import re
import os

def parse_arabic_text(text):
    """
    Parses the structured text (Story, Question, Options) from the
    refined_arabic or initial_arabic fields.

    Args:
        text (str): The string containing the story, question, and options,
                    potentially wrapped in markdown code blocks.

    Returns:
        tuple: A tuple containing (story, question, options_dict),
               or (None, None, {}) if parsing fails.
    """
    if not text or not isinstance(text, str):
        return None, None, {}

    # Remove potential markdown code block fences
    text = re.sub(r'^```[a-zA-Z]*\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n```$', '', text, flags=re.MULTILINE)
    # Remove potential start/end markers like '---'
    text = re.sub(r'^---\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n---\n?$', '', text, flags=re.MULTILINE)
    text = text.strip()

    story = None
    question = None
    options_dict = {}

    try:
        # Use regex to find sections robustly, allowing for variations in spacing/newlines
        # Case-insensitive matching for section headers
        story_match = re.search(r'###\s*STORY\s*(.*?)\s*###\s*QUESTION', text, re.DOTALL | re.IGNORECASE)
        question_match = re.search(r'###\s*QUESTION\s*(.*?)\s*###\s*OPTIONS', text, re.DOTALL | re.IGNORECASE)
        options_match = re.search(r'###\s*OPTIONS\s*(.*)', text, re.DOTALL | re.IGNORECASE)

        if story_match:
            story = story_match.group(1).strip()
            # Further clean story: remove extra newlines within the story if needed
            story = re.sub(r'\n{2,}', '\n', story) # Replace multiple newlines with single

        if question_match:
            question = question_match.group(1).strip()
            question = re.sub(r'\n{2,}', '\n', question) # Clean question text

        if options_match:
            options_text = options_match.group(1).strip()
            # Regex to capture options like (A) Text, (B) Text, etc.
            # Handles potential variations in spacing after the letter and optional periods.
            # Case-insensitive matching for option letters (A-E)
            option_lines = re.findall(r'^\s*\(([A-Ea-e])\)\.?\s*(.*)', options_text, re.MULTILINE | re.IGNORECASE)
            for letter, opt_text in option_lines:
                options_dict[letter.upper()] = opt_text.strip() # Store letter as uppercase

    except Exception as e:
        print(f"Error parsing text block: {e}\nContent sample: {text[:200]}...") # Print first 200 chars on error
        return None, None, {} # Return empty structure on error

    # Basic validation: Ensure essential parts were found if expected
    if not story and "###STORY" in text.upper():
         print(f"Warning: Story marker found but content parsing failed. Text sample: {text[:100]}...")
    if not question and "###QUESTION" in text.upper():
         print(f"Warning: Question marker found but content parsing failed. Text sample: {text[:100]}...")
    if not options_dict and "###OPTIONS" in text.upper():
         print(f"Warning: Options marker found but content parsing failed. Text sample: {text[:100]}...")


    return story, question, options_dict

def convert_jsonl_folder_to_csv(input_folder_path, csv_output_path):
    """
    Converts all JSONL files in a specified folder to a single CSV file.

    Args:
        input_folder_path (str): Path to the folder containing input JSONL files.
        csv_output_path (str): Path to the output CSV file.
    """
    # Define CSV headers - ensure these match the order in csv_row list
    # --- MODIFICATION START: Added ABILITY and INDEX ---
    headers = [
        '#', 'ID', 'Source', 'Country', 'Group', 'Subject', 'Level',
        'Question', 'BackStory', 'Context', 'Answer Key', 'Option 1',
        'Option 2', 'Option 3', 'Option 4', 'Option 5', 'is_few_shot',
        'ABILITY', 'INDEX' # Added new headers
    ]
    # --- MODIFICATION END ---

    global_row_counter = 0 # Counter across all files

    # Check if input folder exists
    if not os.path.isdir(input_folder_path):
        print(f"Error: Input folder not found at '{input_folder_path}'")
        print("Please ensure the folder exists and the path is correct.")
        # Provide specific guidance for Colab if relevant
        try: # Use try-except in case get_ipython is not defined (e.g., running outside Colab/Jupyter)
            if 'google.colab' in str(get_ipython()): # Check if running in Colab
                 print("In Google Colab, make sure the 'data' folder exists in your current directory,")
                 print("or mount your Google Drive and provide the correct path (e.g., '/content/drive/MyDrive/your_folder/data').")
        except NameError:
            pass # Not in an IPython environment
        return

    try:
        # Open the output CSV file for writing
        with open(csv_output_path, 'w', newline='', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(headers) # Write header row once

            # Iterate through all files in the input folder
            print(f"Scanning folder: {input_folder_path}")
            processed_files_count = 0
            skipped_files_count = 0

            for filename in sorted(os.listdir(input_folder_path)): # Sort for consistent processing order
                # Construct the full path to the file
                file_path = os.path.join(input_folder_path, filename)

                # Process only files ending with .jsonl (case-insensitive)
                if filename.lower().endswith('.jsonl') and os.path.isfile(file_path):
                    processed_files_count += 1
                    print(f"Processing file ({processed_files_count}): {filename}...")

                    try:
                        # Open and read the JSONL file line by line
                        with open(file_path, 'r', encoding='utf-8') as infile:
                            file_line_counter = 0
                            for line in infile:
                                file_line_counter += 1
                                global_row_counter += 1
                                try:
                                    # Load JSON data from each line, skip empty lines
                                    line_content = line.strip()
                                    if not line_content:
                                        print(f"   Skipping empty line {file_line_counter} in {filename}.")
                                        global_row_counter -= 1 # Adjust counter as no row written
                                        continue

                                    data = json.loads(line_content)

                                    # --- Data Extraction ---
                                    # Use .get() with default values for safety against missing keys
                                    scenario_id = data.get('scenario_id', '') # Default to empty string if missing

                                    # Get the source file value from the JSON data itself
                                    source_file_from_json = data.get('source_file', '') # Use value from JSON field

                                    # Prioritize refined_arabic, fall back to initial_arabic
                                    arabic_text = data.get('refined_arabic', data.get('initial_arabic', '')) # Default to empty string

                                    # Parse the Arabic text block using the dedicated function
                                    story, question, options = parse_arabic_text(arabic_text)

                                    # Attempt to get the answer key, checking multiple possible field names
                                    answer_key_field_names = ["答案\nANSWER", "answer_key", "correct_answer"]
                                    answer_key = '' # Default value
                                    for key_name in answer_key_field_names:
                                        if key_name in data:
                                            answer_key = data.get(key_name, '')
                                            if answer_key: # Stop if we found a non-empty key
                                                break
                                    # Clean up answer key (e.g., remove surrounding parentheses or whitespace)
                                    if isinstance(answer_key, str):
                                        answer_key = answer_key.strip().strip('()')


                                    # Extract subject from the source_file value within the JSON
                                    subject = source_file_from_json # Start with the value from JSON
                                    if subject and isinstance(subject, str) and subject.lower().endswith('.jsonl'):
                                        # Remove the .jsonl suffix
                                        subject = subject[:-len('.jsonl')]

                                    # --- MODIFICATION START: Extract ABILITY and INDEX ---
                                    ability = data.get('ABILITY', '') # Extract ABILITY, default to empty string
                                    index_val = data.get('INDEX', '')   # Extract INDEX, default to empty string
                                    # --- MODIFICATION END ---


                                    # --- Prepare CSV Row ---
                                    # Ensure the order matches the 'headers' list defined earlier
                                    # --- MODIFICATION START: Appended ability and index_val ---
                                    csv_row = [
                                        global_row_counter,           # '#' (Global counter)
                                        scenario_id,                  # 'ID'
                                        source_file_from_json,        # 'Source' (Value from JSON 'source_file' key)
                                        '',                           # 'Country' (Placeholder)
                                        '',                           # 'Group' (Placeholder)
                                        subject,                      # 'Subject' (Derived from source_file)
                                        '',                           # 'Level' (Placeholder)
                                        question if question else '', # 'Question' (Parsed, default empty)
                                        story if story else '',       # 'BackStory' (Parsed, default empty)
                                        '',                           # 'Context' (Placeholder)
                                        answer_key,                   # 'Answer Key' (Extracted and cleaned)
                                        options.get('A', ''),         # 'Option 1' (Default empty)
                                        options.get('B', ''),         # 'Option 2' (Default empty)
                                        options.get('C', ''),         # 'Option 3' (Default empty)
                                        options.get('D', ''),         # 'Option 4' (Default empty)
                                        options.get('E', ''),         # 'Option 5' (Default empty)
                                        False,                        # 'is_few_shot' (Default to False)
                                        ability,                      # 'ABILITY' (Extracted value)
                                        index_val                     # 'INDEX' (Extracted value)
                                    ]
                                    # --- MODIFICATION END ---

                                    # Write the prepared row to the CSV file
                                    writer.writerow(csv_row)

                                except json.JSONDecodeError:
                                    print(f"   Warning: Skipping invalid JSON on line {file_line_counter} in {filename}. Content: {line_content[:100]}...")
                                except Exception as e:
                                    print(f"   Warning: Error processing line {file_line_counter} in {filename}: {e}")
                                    print(f"   Problematic line content: {line_content[:100]}...")
                                    # Optionally, write a placeholder row or log the error differently

                        print(f"   Finished processing {filename} ({file_line_counter} lines).")

                    except FileNotFoundError:
                         # This shouldn't happen due to the isfile check, but good practice
                        print(f"   Error: Could not open input file '{file_path}' (unexpected). Skipping.")
                        skipped_files_count += 1
                    except Exception as e:
                        print(f"   Error reading file {filename}: {e}. Skipping this file.")
                        skipped_files_count += 1
                elif os.path.isdir(file_path):
                    print(f"Skipping directory: {filename}")
                elif not filename.lower().endswith('.jsonl'):
                     print(f"Skipping non-JSONL file: {filename}")
                     skipped_files_count += 1


            print(f"\n--- Processing Summary ---")
            print(f"Successfully processed {processed_files_count - skipped_files_count} JSONL files.")
            if skipped_files_count > 0:
                print(f"Skipped {skipped_files_count} files/directories.")
            print(f"Total lines written to CSV: {global_row_counter}")
            print(f"CSV output saved to: '{csv_output_path}'")

    except IOError as e:
        print(f"Fatal Error: Could not write to output file '{csv_output_path}': {e}")
    except Exception as e:
        print(f"An unexpected fatal error occurred during processing: {e}")

# --- Script Execution ---
if __name__ == "__main__":
    # --- Configuration ---
    # Define input folder and output file paths
    # Option 1: Relative path (assumes 'data' folder is in the same directory as the script)
    input_folder = 'data'

    # Option 2: Absolute path (replace with your actual path if needed)
    # input_folder = '/path/to/your/data/folder'

    # Option 3: Google Colab - Mount Drive and set path
    # try:
    #     from google.colab import drive
    #     drive.mount('/content/drive')
    #     input_folder = '/content/drive/MyDrive/your_project_folder/data' # Adjust path as needed
    # except ImportError:
    #     print("Not running in Google Colab or drive mounting failed.")
    #     # Keep the default input_folder or handle error as needed
    #     pass


    output_file = 'output_combined_with_ability_index.csv' # Updated output file name

    print("Starting JSONL to CSV conversion...")
    # Run the conversion function
    convert_jsonl_folder_to_csv(input_folder, output_file)
    print("Conversion process finished.")

Starting JSONL to CSV conversion...
Scanning folder: data
Processing file (1): updated_modified (4).jsonl...
   Finished processing updated_modified (4).jsonl (2860 lines).

--- Processing Summary ---
Successfully processed 1 JSONL files.
Total lines written to CSV: 2860
CSV output saved to: 'output_combined_with_ability_index.csv'
Conversion process finished.
